# AIP Petrol Prices (Terminal Gate Prices)

## Setup

In [1]:
# Standard library
import io
import urllib.request
from pathlib import Path
from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from collections.abc import Iterator

import mgplot as mg

# Third party
import pandas as pd

# Local
from abs_structured_capture import ReqsTuple, load_series

In [2]:
# Pandas display
pd.options.display.max_rows = 999999

# Chart output
CHART_DIR = "./CHARTS/AIP/"
mg.set_chart_dir(CHART_DIR)
mg.clear_chart_dir()
SHOW = False
SOURCE = "Source: AIP"

# Date ranges
SHORT_START = pd.Period("2025-12-01", freq="D")

# Cities for per-city charts
CITIES = ["Sydney", "Melbourne", "Brisbane", "Perth"]

# Key event annotations
HORMUZ_CLOSURE = {
    "x": pd.Period("2026-02-28", freq="D").ordinal,
    "color": "grey",
    "linestyle": "--",
    "linewidth": 1,
    "label": "Hormuz closure",
}
EXCISE_CUT = {
    "x": pd.Period("2026-04-01", freq="D").ordinal,
    "color": "grey",
    "linestyle": "-.",
    "linewidth": 1,
    "label": "Fuel Excise Reduction ~32c/l",
}

# AIP download settings
CACHE_DIR = Path("./CACHE/AIP")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
AIP_BASE = "https://www.aip.com.au/sites/default/files/download-files"
AIP_HEADERS = {"User-Agent": "Mozilla/5.0"}

## Helper functions

In [3]:
def _candidate_fridays(n: int = 4) -> Iterator[tuple[str, str]]:
    """Yield (filename, folder) for the most recent n Fridays."""
    today = pd.Timestamp.today().normalize()
    days_since_friday = (today.weekday() - 4) % 7
    friday = today - pd.Timedelta(days=days_since_friday)
    for i in range(n):
        dt = friday - pd.Timedelta(weeks=i)
        folder = dt.strftime("%Y-%m")
        for day_fmt in ("%d", "%-d"):
            filename = f"AIP_TGP_Data_{dt.strftime(f'{day_fmt}-%b-%Y')}.xlsx"
            yield filename, folder


def fetch_tgp_data() -> bytes:
    """Get AIP TGP Excel: check cache then web for each recent Friday."""
    for filename, folder in _candidate_fridays():
        cache_path = CACHE_DIR / filename
        if cache_path.exists():
            print(f"Using cached: {filename}")
            return cache_path.read_bytes()
        url = f"{AIP_BASE}/{folder}/{filename}"
        req = urllib.request.Request(url, headers=AIP_HEADERS)  # noqa: S310
        try:
            data = urllib.request.urlopen(req, timeout=30).read()  # noqa: S310
        except urllib.error.HTTPError:
            continue
        else:
            print(f"Downloaded: {filename}")
            cache_path.write_bytes(data)
            return data
    raise FileNotFoundError("Could not find AIP TGP data file")


def parse_tgp_sheet(raw: bytes, sheet: str) -> pd.DataFrame:
    """Parse an AIP TGP Excel sheet into a clean PeriodIndex DataFrame."""
    df = pd.read_excel(
        io.BytesIO(raw),
        sheet_name=sheet,
        header=0,
        index_col=0,
        parse_dates=True,
    )
    df.columns = df.columns.str.strip().str.replace("\n", " ")
    df.index = pd.DatetimeIndex(df.index).to_period("D")
    return df

In [4]:
def build_footer(df: pd.DataFrame, *, city_level: bool = False) -> str:
    """Build a standard left-footer string for AIP charts."""
    data_to = df.index[-1].strftime("%-d-%b-%Y")
    scope = "business days" if city_level else "national average, business days"
    return (
        f"Australia. Daily wholesale terminal gate prices (inc. GST), "
        f"{scope}. Data to {data_to}."
    )


def get_daily_deflator(lookback_quarters: int = 2) -> tuple[pd.Series, str]:
    """Fetch quarterly CPI, interpolate to daily, project forward to today."""
    cpi_req = ReqsTuple(
        "6401.0", "64010Appendix1a",
        "Index Numbers ;  All groups CPI, seasonally adjusted",
        "S", "", False, False, "",
    )
    cpi_q = load_series(cpi_req)

    recent_growth = cpi_q.pct_change().iloc[-lookback_quarters:].mean()
    today = pd.Timestamp.today().normalize()
    today_q = pd.Period(today, freq="Q")
    while cpi_q.index[-1] <= today_q:
        next_q = cpi_q.index[-1] + 1
        cpi_q[next_q] = cpi_q.iloc[-1] * (1 + recent_growth)

    cpi_daily = (
        cpi_q.to_timestamp(how="end")
        .resample("D")
        .interpolate(method="linear")
    )
    cpi_today = cpi_daily.loc[today]
    deflator = cpi_today / cpi_daily
    deflator.index = deflator.index.to_period("D")

    dollar_label = today.strftime("%B %Y") + " dollar terms"
    return deflator, dollar_label

## Data fetch

In [5]:
def load_fuel_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Fetch and parse petrol and diesel TGP data, return (petrol, diesel, combined)."""
    raw = fetch_tgp_data()
    petrol = parse_tgp_sheet(raw, "Petrol TGP")
    diesel = parse_tgp_sheet(raw, "Diesel TGP")
    combined = pd.DataFrame({
        "Petrol (ULP)": petrol["National Average"],
        "Diesel": diesel["National Average"],
    }).dropna()
    print(f"Petrol: {petrol.index[0]} to {petrol.index[-1]}")
    print(f"Diesel: {diesel.index[0]} to {diesel.index[-1]}")
    return petrol, diesel, combined


petrol, diesel, fuel_prices = load_fuel_data()

Using cached: AIP_TGP_Data_10-Apr-2026.xlsx


Petrol: 2004-01-01 to 2026-04-10
Diesel: 2004-01-01 to 2026-04-10


## National fuel prices

In [6]:
def plot_fuel_prices(fuel: pd.DataFrame) -> None:
    """Plot long-run and short-run national petrol and diesel TGPs."""
    footer = build_footer(fuel)
    common = {
        "title": "Fuel Terminal Gate Prices: Petrol and Diesel",
        "ylabel": "Cents per litre",
        "xlabel": None,
        "legend": {"loc": "best", "fontsize": "x-small"},
        "annotate": True,
        "lfooter": footer,
        "rfooter": SOURCE,
        "show": SHOW,
    }
    # Long run: no event lines
    mg.line_plot_finalise(fuel, **common)
    # Short run: with event lines
    mg.line_plot_finalise(
        fuel.loc[SHORT_START:],
        tag="short",
        axvline=[HORMUZ_CLOSURE, EXCISE_CUT],
        **common,
    )


plot_fuel_prices(fuel_prices)

## City-level fuel prices

In [7]:
def plot_city_fuel(fuel_df: pd.DataFrame, fuel_name: str) -> None:
    """Plot short-run city-level TGPs for a given fuel type."""
    city_data = fuel_df[CITIES].loc[SHORT_START:].dropna()
    mg.line_plot_finalise(
        city_data,
        title=f"{fuel_name} Terminal Gate Prices: Major Cities",
        ylabel="Cents per litre",
        xlabel=None,
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        axvline=[HORMUZ_CLOSURE, EXCISE_CUT],
        lfooter=build_footer(city_data, city_level=True),
        rfooter=SOURCE,
        show=SHOW,
    )


plot_city_fuel(petrol, "Petrol (ULP)")
plot_city_fuel(diesel, "Diesel")

## Inflation-adjusted fuel prices

In [8]:
def plot_real_fuel_prices(fuel: pd.DataFrame) -> None:
    """Plot CPI-deflated long-run petrol and diesel prices."""
    deflator, dollar_label = get_daily_deflator()
    real_fuel = pd.DataFrame({
        "Petrol (ULP)": fuel["Petrol (ULP)"] * deflator,
        "Diesel": fuel["Diesel"] * deflator,
    }).dropna()
    mg.line_plot_finalise(
        real_fuel,
        title="Real Fuel Terminal Gate Prices: Petrol and Diesel",
        ylabel="Cents per litre",
        xlabel=None,
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        lfooter=(
            f"Australia. Daily wholesale TGP (inc. GST). "
            f"In {dollar_label}, using All Groups CPI (SA)."
        ),
        rfooter=f"{SOURCE}, ABS",
        show=SHOW,
    )


plot_real_fuel_prices(fuel_prices)

## Watermark

In [9]:
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-04-12 11:36:03

Python implementation: CPython
Python version       : 3.14.0
IPython version      : 9.12.0

conda environment: n/a

Compiler    : Clang 20.1.4 
OS          : Darwin
Release     : 25.3.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot : 0.2.21
pandas : 3.0.2
pathlib: 1.0.1

Watermark: 2.6.0

